# Viseca sandbox API quickstart

This notebook runs the one-purchase connection check (`SCEN0000`) against the hosted API.

1. Load the key from `.env` and read the team settings.
2. Build a small decision function and test it offline on the example event.
3. Create and confirm a mandate.
4. Start a run, then poll, decide and resolve.

Team reset is disabled (`features.reset = false`), so the mandate and run created in sections 3 and 4 **stay on the team's record**. Sections 1 and 2 only read.

## 1. Connect

In [1]:
import json, time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")  # run the notebook from the repo root
import os
BASE = os.environ["LEASH_BASE_URL"]
S = requests.Session()
S.headers.update({"Authorization": f"Bearer {os.environ['TEAM_API_KEY']}"})

def api(method, path, **kw):
    r = S.request(method, BASE + path, timeout=30, **kw)
    if r.status_code == 204:
        return None
    if not r.ok:
        raise RuntimeError(f"{method} {path} -> {r.status_code}: {r.text}")
    return r.json()

print(requests.get(BASE + "/healthz", timeout=10).json())
boot = api("GET", "/v1/bootstrap")
print("team:", boot["team_id"], "| limits:", boot["limits"], "| features:", boot["features"])
scen = next(s for s in boot["scenarios"] if s["scenario_id"] == "SCEN0000")
INSTRUCTION = scen["cardholder_instruction"]
print("instruction:", INSTRUCTION)

{'status': 'ok', 'service': 'saw26-sandbox', 'api_version': '0.1.0', 'pack_version': 'saw26'}
team: team2 | limits: {'decision_timeout_seconds': 8, 'step_up_timeout_seconds': 120, 'long_poll_max_seconds': 25} | features: {'reset': False}
instruction: Buy one ordinary grocery item for CHF 20 or less from a shop I use regularly. Ask me when uncertain.


## 2. Decision function

The instruction has three parts: price, what is bought, and a shop used regularly. The price and item checks go in `hard_rules`. The API stores these field names without interpreting them, so `check_rule` below is where they get their meaning.

Shop familiarity comes from the history CSV: a shop counts as regular if this card has at least 2 approved purchases there. When a fact can't be established, the mandate's `uncertainty_policy` decides (`ask` means `step_up`).

In [2]:
HARD_RULES = [
    {"field": "authorization.billing_amount_chf", "operator": "<=", "value": 20, "currency": "CHF", "scope": "purchase"},
    {"field": "authorization.items.item_category", "operator": "in", "value": ["groceries"]},
    {"field": "authorization.items.total_quantity", "operator": "<=", "value": 1},
]

hist = pd.read_csv("viseca-2026/data/authorization_history.csv")
approved = hist[hist.status == "approved"]
visits = approved.groupby(["card_id", "merchant_id"]).size()

OPS = {"<": lambda a, b: a < b, "<=": lambda a, b: a <= b, "=": lambda a, b: a == b,
       "!=": lambda a, b: a != b, ">": lambda a, b: a > b, ">=": lambda a, b: a >= b,
       "in": lambda a, b: a in b, "not_in": lambda a, b: a not in b}

def facts(auth, field):
    """Return the list of values a rule field refers to (one per basket line for item fields)."""
    items = auth["items"]
    if field == "authorization.items.item_category":
        return [i["item_category"] for i in items]
    if field == "authorization.items.total_quantity":
        return [sum(i["quantity"] for i in items)]
    key = field.removeprefix("authorization.")
    return [auth.get(key)]

def check_rule(auth, rule):
    vals = facts(auth, rule["field"])
    if any(v is None for v in vals):
        return None  # missing fact: not a pass
    return all(OPS[rule["operator"]](v, rule["value"]) for v in vals)

def decide(event):
    auth, mandate = event["authorization"], event["mandate"]
    base = {"authorization_id": auth["authorization_id"], "engine_version": "quickstart-0.1"}
    if auth["card_status_at_attempt"] != "active" or auth["authority_status"] != "active":
        return {**base, "decision": "decline", "reason_codes": ["card_or_authority_inactive"],
                "customer_message": "The card or authority is not active."}

    failed, unknown = [], []
    for rule in mandate["hard_rules"]:
        ok = check_rule(auth, rule)
        if ok is False:
            failed.append(f"{rule['field']} {rule['operator']} {rule['value']}")
        elif ok is None:
            unknown.append(rule["field"])
    if failed:
        return {**base, "decision": "decline", "reason_codes": ["hard_rule_failed"],
                "customer_message": "Blocked by your rules: " + "; ".join(failed)}

    n = int(visits.get((auth["card_id"], auth["merchant"]["merchant_id"]), 0))
    if n < 2:
        unknown.append(f"shop familiarity ({n} earlier approved purchases)")
    if unknown:
        d = {"ask": "step_up", "decline": "decline", "approve": "approve"}[mandate["uncertainty_policy"]]
        return {**base, "decision": d, "reason_codes": ["uncertain"],
                "customer_message": "Not sure about: " + "; ".join(unknown)}

    return {**base, "decision": "approve", "reason_codes": ["within_mandate"],
            "customer_message": f"CHF {auth['billing_amount_chf']:.2f} at {auth['merchant']['merchant_name']}, "
                                f"a shop used {n} times before."}

Offline test on the parser example (no API calls). Its shop is fake, so expect `step_up`.

In [3]:
example = json.load(open("viseca-2026/data/scenario_fixtures/example_authorization_request.json"))
example["mandate"]["hard_rules"] = HARD_RULES
decide(example)

{'authorization_id': 'AU_EXAMPLE_0001',
 'engine_version': 'quickstart-0.1',
 'decision': 'step_up',
 'reason_codes': ['uncertain'],
 'customer_message': 'Not sure about: shop familiarity (0 earlier approved purchases)'}

## 3. Create and confirm the mandate

This writes to the team's record. Review the rules printed below before confirming.

In [4]:
draft = api("POST", "/v1/mandates", json={
    "instruction": INSTRUCTION,  # exact wording
    "hard_rules": HARD_RULES,
    "uncertainty_policy": "ask",
    "guidance": ["A shop is 'regular' if this card has 2+ approved purchases there in the history file."],
    "open_questions": [],
})
print(json.dumps(draft, indent=2))

{
  "draft_id": "draft_4da9591151cddde9",
  "mandate_id": null,
  "status": "draft",
  "instruction": "Buy one ordinary grocery item for CHF 20 or less from a shop I use regularly. Ask me when uncertain.",
  "hard_rules": [
    {
      "field": "authorization.billing_amount_chf",
      "operator": "<=",
      "value": 20,
      "currency": "CHF",
      "scope": "purchase",
      "period_days": null
    },
    {
      "field": "authorization.items.item_category",
      "operator": "in",
      "value": [
        "groceries"
      ],
      "currency": null,
      "scope": null,
      "period_days": null
    },
    {
      "field": "authorization.items.total_quantity",
      "operator": "<=",
      "value": 1,
      "currency": null,
      "scope": null,
      "period_days": null
    }
  ],
  "uncertainty_policy": "ask",
  "guidance": [
    "A shop is 'regular' if this card has 2+ approved purchases there in the history file."
  ],
  "open_questions": [],
  "created_at": "2026-09-24T11:34:

In [5]:
confirmed = api("POST", f"/v1/mandates/{draft['draft_id']}/confirm", json={"confirmed": True})
MANDATE_ID = confirmed["mandate_id"]
MANDATE_ID

'TMf435f43f2a48d755'

## 4. Run the scenario

The worker polls, decides within the 8-second deadline, and for a `step_up` asks you in the input box (you have 120 s). It stops once every event in the run has been handled.

In [6]:
run = api("POST", "/v1/scenario-runs", json={"scenario_id": "SCEN0000", "mandate_id": MANDATE_ID})
RUN_ID = run["run_id"]
expected = scen["event_count"]
handled = {}

while len(handled) < expected:
    env = api("GET", "/v1/decision-requests/next", params={"wait": 25})
    if env is None:
        print("204, progress:", api("GET", f"/v1/scenario-runs/{RUN_ID}"))
        continue
    event = env["data"]
    auth_id = event["authorization"]["authorization_id"]
    if auth_id in handled:  # redelivery: already decided
        continue
    d = decide(event)
    api("POST", f"/v1/authorizations/{auth_id}/decision", json=d)
    handled[auth_id] = d
    a = event["authorization"]
    print(f"{a['source_authorization_id']}: {d['decision']} | {d['customer_message']}")
    print("  items:", [(i["item_name"], i["quantity"], i["unit_price"]) for i in a["items"]])

    if d["decision"] == "step_up":
        ans = input("Approve this purchase? [y/N] ").strip().lower()
        final = "approve" if ans == "y" else "decline"
        api("POST", f"/v1/authorizations/{auth_id}/resolve", json={
            "decision": final,
            "customer_message": f"The customer chose to {final} this purchase.",
            "evidence": [],
        })
        handled[auth_id] = {**d, "resolved": final}
        print("  customer:", final)

api("GET", f"/v1/scenario-runs/{RUN_ID}")

AU0001: approve | CHF 20.00 at Alpine Basket, a shop used 26 times before.
  items: [('Fresh produce selection', 1, 13.0)]


{'run_id': 'run_347a4523e74f624d',
 'scenario_id': 'SCEN0000',
 'mandate_id': 'TMf435f43f2a48d755',
 'status': 'completed',
 'fixture_profiles': [{'profile_id': 'PROFILE_AUTH0001',
   'customer_id': 'CU0001',
   'card_id': 'CA0001'}],
 'generated_event_count': 1,
 'delivered_event_count': 1,
 'finalized_event_count': 1,
 'processed_event_count': 1,
 'pending_event_count': 0,
 'queued_event_count': 0,
 'platform_rejected_count': 0}

## 5. Inspect results

In [ ]:
pd.json_normalize(api("GET", "/v1/authorizations"))